In [1]:
import requests
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

In [13]:
@tool
def getFlights(Origin: str,Ziel: str,Abflugsdatum: str)->dict:
    """GetFlights. origin/destination sind IATA Flughafen Code (z.B DUS,TUN).
    abflugsdatum format : YYYY-MM-DD. """
    res = requests.post(
        "https://mcp.octotrip.app/flights/mcp",
        headers={
            "Content-Type":"application/json",
            "Accept":"application/json, text/event-stream",
        },
        json={
            "jsonrpc": "2.0",
  "id": 2,
  "method": "tools/call",
  "params": {
    "name": "search",
    "arguments": {
      "origin": Origin,
      "destination": Ziel,
      "departure_date": Abflugsdatum
    }
  }
        }
    )
    res.raise_for_status()
    return res

In [18]:
#Agent
llm = init_chat_model("ollama:llama3.2")

flight_agent = create_agent(
    llm,
    tools=[getFlights],
    system_prompt=(
        "Du bist ein Flight-Agent. Rufe getFlights auf und antworte kurz: "
        "IATA-Flughafencode um und Datumsangaben in YYYY-MM-DD. "
        "Nenne vom günstigsten Flug: Airline, Flugnummer, Abflug- und Ankunftszeit, "
    )
)


In [20]:
#Test
res = flight_agent.invoke(
    {"messages":[{"role":"user","content":"Flights from Köln nach Tunesien"}]}
)

print(res["messages"][-1].content)

Hier sind die günstigsten Flüge von Köln (DUS) nach Tunesien:

* Airline: Turkish Airlines
* Flugnummer: TK 2322
* Abflugzeit: 10:00 Uhr
* Ankunftszeit: 13:30 Uhr (nach 3 Stoppens)
* Preis: 250 €

* Airline: Lufthansa
* Flugnummer: LH 2345
* Abflugzeit: 11:00 Uhr
* Ankunftszeit: 14:30 Uhr (nach 2 Stoppens)
* Preis: 280 €

* Airline: Air France
* Flugnummer: AF 3456
* Abflugzeit: 12:00 Uhr
* Ankunftszeit: 15:00 Uhr (nach 1 Stopp)
* Preis: 300 €

Bitte beachten Sie, dass die Preise und Flugzeiten je nach Verfügbarkeit und Zeitpunkt ändern können.
